# Tech Challenge Fase 3 — Avaliação e Interpretabilidade do Modelo Final

## Objetivo

Avaliar e interpretar o **modelo final enxuto** de alfabetização.

A decisão metodológica é priorizar **parcimônia e explicabilidade**.

### Modelo final

**Regressão Logística**

### Features categóricas

- `rede`
- `sigla_uf`

### Features numéricas

- `log_populacao`
- `vinculos_ativos_por_1000_habitantes`
- `proporcao_vinculos_estatutarios`

### Estratégia temporal

- **2023** → treinamento
- **2024** → teste temporal final *out-of-time*

O hiperparâmetro já foi selecionado no Notebook 04:

`regParam = 0.0`

A região não é utilizada porque é diretamente derivada da UF e adicionaria redundância sem ganho material de desempenho.


## 1. Configurações


In [ ]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import numpy as np
import pandas as pd

CATALOG = "workspace"
GOLD_SCHEMA = "alfabetizacao_gold"

MODEL_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.base_modelagem_aluno"
)

EVALUATION_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.avaliacao_modelo_fase3"
)

COEFFICIENTS_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.coeficientes_modelo_fase3"
)

BEST_REG_PARAM = 0.0

FINAL_MODEL_NAME = (
    "logistica_enxuta"
)

print("Base:", MODEL_TABLE)
print("Modelo final:", FINAL_MODEL_NAME)
print("regParam final:", BEST_REG_PARAM)


## 2. Leitura da Gold e feature engineering


In [ ]:
df_raw = spark.table(MODEL_TABLE)

df = (
    df_raw
    .withColumn(
        "rede",
        F.col("rede").cast("string"),
    )
    .withColumn(
        "log_populacao",
        F.log1p(
            F.col("populacao").cast("double")
        ),
    )
    .withColumn(
        "proporcao_vinculos_estatutarios",
        F.when(
            F.col("quantidade_vinculos_ativos") > 0,
            (
                F.col("quantidade_vinculos_estatutarios")
                / F.col("quantidade_vinculos_ativos")
            ),
        ).otherwise(F.lit(0.0)),
    )
)

df_train = (
    df
    .filter(
        F.col("grupo_modelagem")
        == "DESENVOLVIMENTO_2023"
    )
)

df_test = (
    df
    .filter(
        F.col("grupo_modelagem")
        == "TESTE_TEMPORAL_2024"
    )
)

print(
    "Treino 2023:",
    df_train.count(),
)

print(
    "Teste 2024:",
    df_test.count(),
)


## 3. Features do modelo final

O modelo utiliza apenas cinco features:

### Categóricas
- `rede`
- `sigla_uf`

### Numéricas
- `log_populacao`
- `vinculos_ativos_por_1000_habitantes`
- `proporcao_vinculos_estatutarios`

Essa seleção evita identificadores de alta cardinalidade, variáveis operacionais e redundância entre UF e região.


In [ ]:
categorical_cols = [
    "rede",
    "sigla_uf",
]

numeric_cols = [
    "log_populacao",
    "vinculos_ativos_por_1000_habitantes",
    "proporcao_vinculos_estatutarios",
]

feature_cols = (
    categorical_cols
    + numeric_cols
)

print(
    "Features finais:",
    feature_cols,
)


## 4. Validação de completude

Nenhuma das features utilizadas pelo modelo deve possuir valores ausentes.


In [ ]:
expressoes_nulos = [
    F.sum(
        F.when(
            F.col(c).isNull(),
            1,
        ).otherwise(0)
    ).alias(f"{c}_nulos")
    for c in feature_cols
]

display(
    df
    .groupBy("ano")
    .agg(*expressoes_nulos)
    .orderBy("ano")
)


## 5. Construção da Pipeline

Todo o preprocessing permanece integrado à Pipeline.

### Categóricas
- `StringIndexer`
- `OneHotEncoder`
- categorias não vistas tratadas com `handleInvalid="keep"`

### Numéricas
- montagem em vetor
- padronização por desvio-padrão

### Modelo
- Regressão Logística
- `regParam = 0.0`


In [ ]:
indexers = [
    StringIndexer(
        inputCol=coluna,
        outputCol=f"{coluna}_idx",
        handleInvalid="keep",
    )
    for coluna in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[
        f"{c}_idx"
        for c in categorical_cols
    ],
    outputCols=[
        f"{c}_ohe"
        for c in categorical_cols
    ],
    handleInvalid="keep",
)

numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_features",
)

numeric_scaler = StandardScaler(
    inputCol="numeric_features",
    outputCol="numeric_scaled",
    withStd=True,
    withMean=False,
)

final_assembler = VectorAssembler(
    inputCols=[
        "rede_ohe",
        "sigla_uf_ohe",
        "numeric_scaled",
    ],
    outputCol="features",
)

lr = LogisticRegression(
    labelCol="target_alfabetizado",
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    elasticNetParam=0.0,
    regParam=BEST_REG_PARAM,
    standardization=False,
)

pipeline = Pipeline(
    stages=[
        *indexers,
        encoder,
        numeric_assembler,
        numeric_scaler,
        final_assembler,
        lr,
    ]
)


## 6. Treinamento final em 2023


In [ ]:
final_model = (
    pipeline
    .fit(df_train)
)

print(
    "Modelo final treinado em todo o ano de 2023."
)


## 7. Predição no teste temporal de 2024


In [ ]:
pred_test = (
    final_model
    .transform(df_test)
)

print(
    "Registros avaliados:",
    pred_test.count(),
)


## 8. Matriz de confusão e métricas globais

A classe positiva é:

`1 = alfabetizado`

Além de Accuracy, Precision, Recall, F1 e ROC-AUC, também serão calculadas:

- Specificity
- Balanced Accuracy
- Macro-F1


In [ ]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="target_alfabetizado",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

confusao = (
    pred_test
    .agg(
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("fn"),
    )
    .first()
)

tp = int(confusao["tp"])
tn = int(confusao["tn"])
fp = int(confusao["fp"])
fn = int(confusao["fn"])

total = tp + tn + fp + fn

accuracy = (
    (tp + tn) / total
)

precision = (
    tp / (tp + fp)
    if (tp + fp)
    else 0.0
)

recall = (
    tp / (tp + fn)
    if (tp + fn)
    else 0.0
)

specificity = (
    tn / (tn + fp)
    if (tn + fp)
    else 0.0
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if (precision + recall)
    else 0.0
)

precision_0 = (
    tn / (tn + fn)
    if (tn + fn)
    else 0.0
)

f1_0 = (
    2 * precision_0 * specificity
    / (precision_0 + specificity)
    if (precision_0 + specificity)
    else 0.0
)

balanced_accuracy = (
    (recall + specificity) / 2
)

macro_f1 = (
    (f1 + f1_0) / 2
)

roc_auc = (
    auc_evaluator
    .evaluate(pred_test)
)

metricas = [
    ("accuracy", float(accuracy)),
    ("precision", float(precision)),
    ("recall", float(recall)),
    ("specificity", float(specificity)),
    ("f1", float(f1)),
    ("balanced_accuracy", float(balanced_accuracy)),
    ("macro_f1", float(macro_f1)),
    ("roc_auc", float(roc_auc)),
]

display(
    spark.createDataFrame(
        metricas,
        [
            "metrica",
            "valor",
        ],
    )
)

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)


## 9. Baseline da classe majoritária

O modelo será comparado com a regra mais simples possível:

> sempre prever a classe majoritária observada em 2023.


In [ ]:
majority_class = (
    df_train
    .groupBy("target_alfabetizado")
    .count()
    .orderBy(
        F.desc("count")
    )
    .first()["target_alfabetizado"]
)

baseline_test = (
    df_test
    .withColumn(
        "prediction_baseline",
        F.lit(
            float(majority_class)
        ),
    )
)

baseline_row = (
    baseline_test
    .agg(
        F.sum(
            F.when(
                F.col("target_alfabetizado")
                == F.col("prediction_baseline"),
                1,
            ).otherwise(0)
        ).alias("acertos"),
        F.count("*").alias("total"),
    )
    .first()
)

baseline_accuracy = (
    baseline_row["acertos"]
    / baseline_row["total"]
)

print(
    "Classe majoritária:",
    majority_class,
)

print(
    "Accuracy baseline:",
    round(
        baseline_accuracy,
        4,
    ),
)

print(
    "Ganho absoluto de accuracy:",
    round(
        accuracy - baseline_accuracy,
        4,
    ),
)


## 10. Métricas por UF

Essa análise permite identificar onde o modelo generaliza melhor ou pior territorialmente.


In [ ]:
uf_metrics = (
    pred_test
    .groupBy("sigla_uf")
    .agg(
        F.count("*").alias("alunos"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("fn"),
    )
    .withColumn(
        "accuracy",
        (
            F.col("tp")
            + F.col("tn")
        ) / F.col("alunos"),
    )
    .withColumn(
        "recall_alfabetizado",
        F.when(
            (
                F.col("tp")
                + F.col("fn")
            ) > 0,
            F.col("tp")
            / (
                F.col("tp")
                + F.col("fn")
            ),
        ),
    )
    .withColumn(
        "recall_nao_alfabetizado",
        F.when(
            (
                F.col("tn")
                + F.col("fp")
            ) > 0,
            F.col("tn")
            / (
                F.col("tn")
                + F.col("fp")
            ),
        ),
    )
    .withColumn(
        "balanced_accuracy",
        (
            F.col("recall_alfabetizado")
            + F.col("recall_nao_alfabetizado")
        ) / 2,
    )
)

display(
    uf_metrics
    .select(
        "sigla_uf",
        "alunos",
        F.round(
            "accuracy",
            4,
        ).alias("accuracy"),
        F.round(
            "balanced_accuracy",
            4,
        ).alias("balanced_accuracy"),
        F.round(
            "recall_alfabetizado",
            4,
        ).alias("recall_alfabetizado"),
        F.round(
            "recall_nao_alfabetizado",
            4,
        ).alias("recall_nao_alfabetizado"),
    )
    .orderBy(
        F.desc(
            "balanced_accuracy"
        )
    )
)


## 11. Análise dos erros

Os erros são divididos em:

- `FP`: previsto alfabetizado, mas não alfabetizado;
- `FN`: previsto não alfabetizado, mas alfabetizado;
- `ACERTO`: classificação correta.


In [ ]:
pred_erros = (
    pred_test
    .withColumn(
        "tipo_resultado",
        F.when(
            F.col("prediction")
            == F.col("target_alfabetizado"),
            F.lit("ACERTO"),
        )
        .when(
            (F.col("prediction") == 1)
            & (F.col("target_alfabetizado") == 0),
            F.lit("FP"),
        )
        .otherwise(
            F.lit("FN"),
        ),
    )
)

display(
    pred_erros
    .groupBy(
        "tipo_resultado",
    )
    .agg(
        F.count("*").alias("alunos"),
        F.round(
            F.avg("populacao"),
            2,
        ).alias("populacao_media"),
        F.round(
            F.avg(
                "vinculos_ativos_por_1000_habitantes"
            ),
            2,
        ).alias("vinculos_por_1000_media"),
        F.round(
            F.avg(
                "proporcao_vinculos_estatutarios"
            ),
            4,
        ).alias("proporcao_estatutarios_media"),
    )
    .orderBy(
        F.desc("alunos")
    )
)


## 12. Coeficientes e odds ratios

Na Regressão Logística:

`odds_ratio = exp(coeficiente)`

A interpretação é associativa, não causal.

As features numéricas foram padronizadas. Portanto, seus odds ratios correspondem aproximadamente a uma variação de **1 desvio-padrão na variável transformada**.


In [ ]:
lr_model = (
    final_model
    .stages[-1]
)

coefs = (
    lr_model
    .coefficients
    .toArray()
)

sample_transformed = (
    final_model
    .transform(
        df_train.limit(1)
    )
)

metadata = (
    sample_transformed
    .schema["features"]
    .metadata
)

attrs = (
    metadata["ml_attr"]["attrs"]
)

flattened = []

for attr_type in attrs:
    flattened.extend(
        attrs[attr_type]
    )

flattened = sorted(
    flattened,
    key=lambda x: x["idx"],
)

feature_names = [
    item["name"]
    for item in flattened
]

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coeficiente": coefs,
    }
)

numeric_name_map = {
    "numeric_scaled_0": "log_populacao",
    "numeric_scaled_1": (
        "vinculos_ativos_por_1000_habitantes"
    ),
    "numeric_scaled_2": (
        "proporcao_vinculos_estatutarios"
    ),
}

coef_df["feature"] = (
    coef_df["feature"]
    .replace(
        numeric_name_map
    )
)

coef_df["odds_ratio"] = (
    np.exp(
        coef_df["coeficiente"]
    )
)

coef_df["abs_coeficiente"] = (
    coef_df["coeficiente"]
    .abs()
)

coef_df = (
    coef_df
    .sort_values(
        "abs_coeficiente",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    coef_df.head(30)
)


## 13. Principais associações positivas e negativas


In [ ]:
print(
    "Maiores associações positivas:"
)

display(
    coef_df
    .sort_values(
        "coeficiente",
        ascending=False,
    )
    .head(10)
)

print(
    "Maiores associações negativas:"
)

display(
    coef_df
    .sort_values(
        "coeficiente",
        ascending=True,
    )
    .head(10)
)


## 14. Persistência das métricas finais


In [ ]:
evaluation_rows = [
    (
        FINAL_MODEL_NAME,
        2023,
        2024,
        metrica,
        valor,
    )
    for metrica, valor
    in metricas
]

evaluation_rows.append(
    (
        "baseline_majority",
        2023,
        2024,
        "accuracy",
        float(
            baseline_accuracy
        ),
    )
)

df_evaluation = (
    spark.createDataFrame(
        evaluation_rows,
        [
            "modelo",
            "ano_treino",
            "ano_teste",
            "metrica",
            "valor",
        ],
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp(),
    )
)

(
    df_evaluation
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        EVALUATION_TABLE
    )
)

display(
    spark.table(
        EVALUATION_TABLE
    )
)


## 15. Persistência dos coeficientes


In [ ]:
coef_spark = (
    spark.createDataFrame(
        coef_df[
            [
                "feature",
                "coeficiente",
                "odds_ratio",
                "abs_coeficiente",
            ]
        ]
    )
    .withColumn(
        "modelo",
        F.lit(
            FINAL_MODEL_NAME
        ),
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp(),
    )
)

(
    coef_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        COEFFICIENTS_TABLE
    )
)

display(
    spark.table(
        COEFFICIENTS_TABLE
    )
    .orderBy(
        F.desc(
            "abs_coeficiente"
        )
    )
    .limit(30)
)


## Conclusão

O projeto adota como modelo final uma **Regressão Logística enxuta**, utilizando apenas cinco features.

### Features finais

- `rede`
- `sigla_uf`
- `log_populacao`
- `vinculos_ativos_por_1000_habitantes`
- `proporcao_vinculos_estatutarios`

### Justificativa

A escolha prioriza:

- simplicidade;
- interpretabilidade;
- baixo custo computacional;
- ausência de redundância entre UF e região;
- facilidade de comunicação dos resultados.

### Limitação principal

As features disponíveis são predominantemente territoriais e contextuais.

Por isso, o modelo possui maior capacidade de identificar **contextos associados ao risco de não alfabetização** do que distinguir estudantes individualmente dentro de um mesmo território.

O modelo deve ser utilizado como ferramenta de **inteligência analítica e territorial**, e não como mecanismo causal ou de decisão individual de alto impacto.

### Próximos entregáveis

- README final;
- relatório executivo;
- visualizações para apresentação;
- roteiro do vídeo.
